# Notebook 01 — Engine Sanity Checks

Verifies the matching engine with hand-constructed sequences.

In [ ]:
import sys; sys.path.insert(0, '..')
from lob_sim import LOBBook, Side
import pandas as pd


## 1. Build a symmetric ladder and inspect the snapshot

In [ ]:
LOBBook.debug = True
book = LOBBook()
for i in range(5):
    book.submit_limit(Side.BID, 1000 - 2*(i+1), 10*(i+1), i+1, 0.0)
    book.submit_limit(Side.ASK, 1000 + 2*(i+1), 10*(i+1), 100+i+1, 0.0)

snap = book.snapshot(depth=5)
print(f'Best bid: {book.best_bid()}  Best ask: {book.best_ask()}  Mid: {book.mid()}  Spread: {book.spread()}')
bids_df = pd.DataFrame(snap.bids, columns=['price', 'qty'])
asks_df = pd.DataFrame(snap.asks, columns=['price', 'qty'])
print('\nBids (best first):')
print(bids_df.to_string(index=False))
print('\nAsks (best first):')
print(asks_df.to_string(index=False))

## 2. Crossing limit order — immediate fill

In [ ]:
book2 = LOBBook()
book2.submit_limit(Side.ASK, 100, 10, 1, 0.0)
book2.submit_limit(Side.ASK, 101, 10, 2, 0.0)
book2.submit_limit(Side.ASK, 102, 10, 3, 0.0)

# A crossing bid at 102 should sweep all three levels
trades = book2.submit_limit(Side.BID, 102, 25, 99, 1.0)
print('Trades from crossing bid:')
for t in trades:
    print(f'  passive_oid={t.passive_order_id}  price={t.price_ticks}  qty={t.qty}')
print(f'Total filled: {sum(t.qty for t in trades)}')
print(f'Remaining qty on order 3: {book2.order_qty(3)}')

## 3. Market order walking multiple levels

In [ ]:
book3 = LOBBook()
for i in range(10):
    book3.submit_limit(Side.ASK, 1002 + i*2, 5, i+1, 0.0)

print(f'Book before: {book3.snapshot(3).asks}')
trades = book3.submit_market(Side.BID, 22, 999, 1.0)
print(f'Filled {sum(t.qty for t in trades)} across {len(trades)} levels')
print(f'Prices hit: {[t.price_ticks for t in trades]}')
print(f'Book after: {book3.snapshot(3).asks}')

## 4. FIFO at the same price level

In [ ]:
book4 = LOBBook()
# Three orders at price 100 in time order
for oid, qty in [(1, 3), (2, 5), (3, 2)]:
    book4.submit_limit(Side.ASK, 100, qty, oid, float(oid))

# A market buy of 6 should fill oid=1 fully, oid=2 partially
trades = book4.submit_market(Side.BID, 6, 999, 5.0)
print('Fills (should be oid=1 then oid=2):')
for t in trades:
    print(f'  passive_oid={t.passive_order_id}  qty={t.qty}')
assert trades[0].passive_order_id == 1
assert trades[1].passive_order_id == 2
print('FIFO verified ✓')

## 5. Cancel partially-filled order

In [ ]:
book5 = LOBBook()
book5.submit_limit(Side.ASK, 100, 10, 1, 0.0)
# Partial fill: buy 4
book5.submit_market(Side.BID, 4, 2, 1.0)
print(f'Remaining qty after partial fill: {book5.order_qty(1)}')
# Now cancel the rest
ok = book5.cancel(1, ts=2.0)
print(f'Cancel succeeded: {ok}')
print(f'Order still in book: {book5.has_order(1)}')
print(f'Best ask: {book5.best_ask()} (should be None)')